# 04 Robustness checks and horizon study

Needs the outputs of notebooks 01 to 03. Each section saves its own results file. Sections:
3. Weather mix of validation and test days
4. Adaptive calibration by recent variability
5. A5 against A4 on validation loss
6. Inference time
7. Diffusion with and without clipping during sampling
8. Models trained with the uncorrected clock
9. Forecast horizons of 5, 10, 15 and 30 minutes
10. Diffusion trained for longer, three seeds
11. Models with cloud motion (only if A5 is better on validation)

In [ ]:
!pip install -q datasets pvlib
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/UROP'
LOCAL_CACHE = '/content/skippd_cache'

Mounted at /content/drive


In [ ]:
import os, shutil, time, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
os.makedirs(f'{OUT}/figs', exist_ok=True); os.makedirs(f'{OUT}/ckpt', exist_ok=True)
os.makedirs(f'{OUT}/cache', exist_ok=True); os.makedirs(LOCAL_CACHE, exist_ok=True)
SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
INPUT_LEN, HORIZON = 15, 15
LAT, LON, ALT = 37.4275, -122.1697, 23
PANEL_TILT, PANEL_AZIMUTH, PANEL_KW_DC = 22.5, 195, 30.1
PV_CAP_PCT = 99.95
RUN = 'day4'
TRAIN = dict(batch=256, lr=1e-3, max_epochs=25, patience=4, val_frac_days=0.15)

SEL_NAME = 'A4 + PV variability'
SEL_C = dict(geo=True, cs=True, kout=True, var=True)
A5_C = dict(geo=True, cs=True, kout=True, var=True, motion=True)
ENSEMBLE_SEEDS = [0, 1, 2, 3, 4]
RESID_SEEDS = [0, 1, 2]
A5_SEEDS = [0, 1, 2]
MC_PASSES = 30
DIFF = dict(T=100, max_epochs=15, patience=4, lr=2e-4, batch=512, n_samples=50)
ALPHA = 0.10
RESERVE_LEVEL = 0.95

VAR_BIN_QUANTILES = [0.4, 0.7, 0.9]
DIFF_SEEDS = [0, 1, 2]
DIFF_MODES = ['resid', 'direct']
DIFF_LONG = dict(max_epochs=60, patience=6)
HORIZONS = [5, 10, 30]
RUN_A5_MODELS = True
FORCE_A5 = None

RESULTS_CSV = f'{OUT}/results.csv'
RESULT_COLS = ['when','run','train_data','test_set','model','n','rmse','mae','skill_rmse_vs_smart','skill_mae_vs_smart','notes']
def log_result(**row):
    row['when'] = pd.Timestamp.now().isoformat(timespec='seconds')
    pd.DataFrame([{c: row.get(c, '') for c in RESULT_COLS}]).to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items() if k not in ('when', 'run')})

## 1. Data, clock correction, features and models

In [ ]:
def load_split(split):
    names = ['img', 'pv', 'ts']
    local = {k: f'{LOCAL_CACHE}/{split}_{k}.npy' for k in names}
    drive_ = {k: f'{OUT}/cache/{split}_{k}.npy' for k in names}

    if not all(os.path.exists(p) for p in local.values()) and all(os.path.exists(p) for p in drive_.values()):
        print(f'[{split}] copying cache from Drive to local disk...')
        for k in names: shutil.copy(drive_[k], local[k])

    if all(os.path.exists(p) for p in local.values()):
        print(f'[{split}] loading from cache')
        img = np.load(local['img']); pv = np.load(local['pv']); ts_raw = np.load(local['ts'])
    else:
        from datasets import load_dataset
        print(f'[{split}] downloading from Hugging Face (first run only)...')
        ds = load_dataset('solarbench/SKIPPD', split=split)
        n = len(ds)
        img = np.empty((n, 64, 64, 3), dtype=np.uint8)
        pos = 0
        t0 = time.time()
        for b in ds.iter(batch_size=2000):
            arr = np.stack([np.asarray(im.convert('RGB')) for im in b['image']])
            img[pos:pos + len(arr)] = arr
            pos += len(arr)
            if pos % 50000 < 2000: print(f'  {pos}/{n} images  ({time.time()-t0:.0f}s)')
        pv = np.asarray(ds['pv'], dtype=np.float32)
        t = pd.to_datetime(ds['time'])
        if t.tz is not None:
            t = t.tz_convert('UTC').tz_localize(None)
        ts_raw = t.values.astype('datetime64[ns]')
        for k, v in [('img', img), ('pv', pv), ('ts', ts_raw)]:
            np.save(local[k], v); shutil.copy(local[k], drive_[k])
        print(f'[{split}] cached to Drive')

    order = np.argsort(ts_raw, kind='stable')
    return dict(name=split, img=img, img_idx=order, pv=pv[order].astype(np.float32),
                ts=pd.DatetimeIndex(ts_raw[order]))

In [ ]:
TR = load_split('train'); TE = load_split('test')
prep_path = f'{OUT}/day1b_prep.npz'
assert os.path.exists(prep_path), 'Run notebook 01 first: it creates day1b_prep.npz'
P = np.load(prep_path, allow_pickle=True)
TR['w_clean'] = P['tr_w_clean']; TE['w_clean'] = P['te_w_clean']; TE['w_raw'] = P['te_w_raw']
print('train clean windows:', len(TR['w_clean']), '| test clean:', len(TE['w_clean']), '| test raw:', len(TE['w_raw']))

In [ ]:
from pvlib.location import Location
from pvlib import irradiance
SITE = Location(LAT, LON, altitude=ALT)
TZ = 'UTC'

def solar_terms(ts_naive, tz='UTC'):
    idx = pd.DatetimeIndex(ts_naive)
    aware = idx.tz_localize('UTC') if tz == 'UTC' else idx.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
    ok = ~aware.isna()
    out = [np.full(len(idx), np.nan, dtype=np.float32) for _ in range(3)]
    a = aware[ok]
    cs = SITE.get_clearsky(a, model='ineichen'); sp = SITE.get_solarposition(a)
    poa = irradiance.get_total_irradiance(PANEL_TILT, PANEL_AZIMUTH, dni=cs['dni'], ghi=cs['ghi'], dhi=cs['dhi'],
                                          solar_zenith=sp['apparent_zenith'], solar_azimuth=sp['azimuth'])
    out[0][ok] = np.clip(PANEL_KW_DC * poa['poa_global'].values / 1000.0, 0, PANEL_KW_DC)
    out[1][ok] = sp['apparent_zenith'].values
    out[2][ok] = sp['azimuth'].values
    return out

def local_date(ts):
    return np.array((pd.DatetimeIndex(ts) - pd.Timedelta(hours=8)).date)

def is_dst(d):
    return bool((pd.Timestamp(d) + pd.Timedelta(hours=12)).tz_localize('America/Los_Angeles').dst())

FIX_PATH = f'{OUT}/clock_fix.json'
if os.path.exists(FIX_PATH):
    CLOCK_FIX = json.load(open(FIX_PATH)); print('clock fix loaded:', CLOCK_FIX)
else:
    ld = local_date(TR['ts'])
    df = pd.DataFrame({'pv': TR['pv'], 'd': ld}, index=TR['ts'])
    rough = {d: np.abs(np.diff(g.pv.values)).sum() / max(g.pv.sum(), 1e-6)
             for d, g in df.groupby('d') if len(g) >= 300 and g.pv.max() >= 15}
    rough = pd.Series(rough).sort_values()
    picks = pd.DataFrame({'day': rough.index}).assign(m=lambda x: [(d.year, d.month) for d in x.day]).groupby('m').head(3).day
    rows = []
    for d in picks:
        p = df.pv[ld == d]; p.index = p.index.floor('min'); p = p[~p.index.duplicated()]
        grid = pd.date_range(p.index.min() - pd.Timedelta(minutes=150), p.index.max() + pd.Timedelta(minutes=150), freq='1min')
        cs = pd.Series(solar_terms(grid)[0], index=grid); pv_g = p.reindex(grid)
        best = (0, -2.0)
        for L in range(-120, 125, 5):
            c = cs.shift(L); ok = pv_g.notna() & c.notna()
            if ok.sum() < 100: continue
            r = np.corrcoef(pv_g[ok], c[ok])[0, 1]
            if r > best[1]: best = (L, r)
        rows.append(dict(day=d, dst=is_dst(d), lag=best[0], corr=best[1]))
    lag = pd.DataFrame(rows); good = lag[lag['corr'] >= 0.95]
    m_dst, m_std = good[good.dst].lag.median(), good[~good.dst].lag.median()
    print(f'{len(good)}/{len(lag)} clear days usable. Median shift: daylight saving {m_dst} min, standard {m_std} min')
    std = int(5 * round(m_std / 5)) if abs(m_std) >= 15 else 0
    CLOCK_FIX = {'dst_min': std + 60 * int(round((m_dst - m_std) / 60)), 'std_min': std}
    json.dump(CLOCK_FIX, open(FIX_PATH, 'w')); lag.to_csv(f'{OUT}/clock_offset_by_day_v2.csv', index=False)
    print('clock fix:', CLOCK_FIX)

def solar_time(ts):
    ts = pd.DatetimeIndex(ts); ld = local_date(ts)
    lut = {d: is_dst(d) for d in set(ld)}
    shift = np.where(np.array([lut[d] for d in ld]), CLOCK_FIX['dst_min'], CLOCK_FIX['std_min'])
    return ts - pd.to_timedelta(shift, unit='min')

for D in (TR, TE):
    cs, zen, azi = solar_terms(solar_time(D['ts']))
    D['cs_raw'] = np.nan_to_num(cs); D['zen'] = np.nan_to_num(zen, nan=180.0); D['azi'] = np.nan_to_num(azi)
PV_CAP = float(np.percentile(TR['pv'], PV_CAP_PCT))
for D in (TR, TE):
    D['cs'] = np.minimum(D['cs_raw'], PV_CAP).astype(np.float32)
    D['k'] = np.clip(D['pv'] / np.maximum(D['cs'], 1.0), 0, 1.5).astype(np.float32)
    doy = D['ts'].dayofyear.values
    D['doy_s'] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32); D['doy_c'] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)
    D['ld'] = local_date(D['ts'])
print(f'PV cap (inverter limit estimate): {PV_CAP:.2f} kW')

In [ ]:
def metrics(pred, true):
    return float(np.sqrt(np.mean((pred - true) ** 2))), float(np.mean(np.abs(pred - true)))

PRED = {}
SIGMA = {}
TESTS = {'raw test': TE['w_raw'], 'clean test': TE['w_clean']}
for label, w in TESTS.items():
    last = TE['pv'][w]
    PRED[('naive persistence', label)] = last
    PRED[('smart persistence', label)] = np.clip(last / np.maximum(TE['cs_raw'][w], 1e-3) * TE['cs_raw'][w + HORIZON], 0, None)
BASE = {label: metrics(PRED[('smart persistence', label)], TE['pv'][w + HORIZON]) for label, w in TESTS.items()}
for label, (r, m) in BASE.items(): print(f'smart persistence on {label}: RMSE {r:.3f}  MAE {m:.3f}')

def score(name, notes=''):
    for label, w in TESTS.items():
        y = TE['pv'][w + HORIZON]; r, m = metrics(PRED[(name, label)], y); r_s, m_s = BASE[label]
        log_result(run=RUN, train_data='clean', test_set=label, model=name, n=len(y), rmse=r, mae=m,
                   skill_rmse_vs_smart=1 - r / r_s, skill_mae_vs_smart=1 - m / m_s, notes=notes)
score('naive persistence'); score('smart persistence')

In [ ]:
OFF = np.arange(-(INPUT_LEN - 1), 1)

def vec_feats(D, t, c):
    H = t[:, None] + OFF; tg = t + HORIZON
    parts = [D['pv'][H] / PV_CAP]
    if c.get('geo'):
        parts.append(np.stack([D['doy_s'][t], D['doy_c'][t], D['zen'][t] / 90, D['zen'][tg] / 90,
                               np.cos(np.radians(D['zen'][t])), np.sin(np.radians(D['azi'][t])), np.cos(np.radians(D['azi'][t]))], 1))
    if c.get('cs'):
        parts += [D['cs'][H] / PV_CAP, (D['cs'][tg] / PV_CAP)[:, None]]
    if c.get('kout'):
        parts.append(D['k'][H])
    if c.get('var'):
        kh, ph = D['k'][H], D['pv'][H]
        parts.append(np.stack([kh.std(1), np.abs(np.diff(ph, axis=1)).mean(1) / PV_CAP,
                               (ph[:, -1] - ph[:, -6]) / PV_CAP, (ph.max(1) - ph.min(1)) / PV_CAP], 1))
    return np.concatenate(parts, 1).astype(np.float32)

def to_img(D, t):
    return torch.from_numpy(D['img'][D['img_idx'][t]]).to(device).permute(0, 3, 1, 2).float().div_(255.0)

def batches(D, w, c, bs, shuffle, rng=None):
    idx = rng.permutation(len(w)) if shuffle else np.arange(len(w))
    for i in range(0, len(w), bs):
        t = w[idx[i:i + bs]]
        xv = torch.from_numpy(vec_feats(D, t, c)).to(device)
        img = to_img(D, t)
        if c.get('motion'): img = torch.cat([img, img - to_img(D, t - 5)], 1)
        cst = torch.from_numpy(D['cs'][t + HORIZON] / PV_CAP).float().to(device)
        y = torch.from_numpy(D['pv'][t + HORIZON] / PV_CAP).float().to(device)
        yield xv, img, cst, y

class FeatCNN(nn.Module):
    '''Same image branch as the notebook 01 CNN; the PV branch takes the extra feature vector.
       With kout=True it outputs a clear-sky index and the forecast is index x clear-sky PV at the target time.'''
    def __init__(self, n_vec, img_ch=3, use_image=True, kout=False, p_drop=0.2):
        super().__init__()
        self.use_image, self.kout = use_image, kout
        if use_image:
            self.conv = nn.Sequential(
                nn.Conv2d(img_ch, 16, 5, stride=2, padding=2), nn.ReLU(),
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),
                nn.AdaptiveAvgPool2d(4))
        self.vec = nn.Sequential(nn.Linear(n_vec, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Sequential(nn.Linear((32 * 16 if use_image else 0) + 32, 64), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(64, 1))
    def forward(self, xv, img, cst):
        f = self.vec(xv)
        if self.use_image: f = torch.cat([self.conv(img).flatten(1), f], 1)
        out = self.head(f).squeeze(-1)
        return out * cst if self.kout else out

def n_vec(c):
    return vec_feats(TR, TR['w_clean'][:2], c).shape[1]

def split_by_days(D, w, frac, seed=SEED):
    days = D['ld'][w]; uniq = np.array(sorted(set(days)))
    val_days = set(np.random.default_rng(seed).choice(uniq, int(len(uniq) * frac), replace=False))
    m = pd.Series(days).isin(val_days).values
    return w[~m], w[m]
W_TR, W_VA = split_by_days(TR, TR['w_clean'], TRAIN['val_frac_days'])
print(f'{len(W_TR)} train / {len(W_VA)} val windows (validation = random 15% of days, same as notebook 01)')

In [ ]:
def smart_base(D, t):
    '''Smart persistence, capped at the inverter limit. Used as the starting point of the correction models.'''
    return np.clip(D['pv'][t] / np.maximum(D['cs_raw'][t], 1e-3) * D['cs_raw'][t + HORIZON], 0, PV_CAP).astype(np.float32)

_vec_feats_day2 = vec_feats
def vec_feats(D, t, c):
    x = _vec_feats_day2(D, t, c)
    if c.get('resid'):
        x = np.concatenate([x, (smart_base(D, t) / PV_CAP)[:, None]], 1)
    return x

class ResidCNN(FeatCNN):
    '''Forecast = smart persistence + learned correction.'''
    def __init__(self, n_vec, img_ch=3, use_image=True):
        super().__init__(n_vec, img_ch=img_ch, use_image=use_image, kout=False)
    def forward(self, xv, img, cst):
        return xv[:, -1] + super().forward(xv, img, cst)

def make_model(c, use_image=True):
    kw = dict(img_ch=6 if c.get('motion') else 3, use_image=use_image)
    m = ResidCNN(n_vec(c), **kw) if c.get('resid') else FeatCNN(n_vec(c), kout=bool(c.get('kout')), **kw)
    return m.to(device)

for label, w in TESTS.items():
    PRED[('smart persistence (capped)', label)] = smart_base(TE, w)
score('smart persistence (capped)', notes='capped at inverter limit; the starting point of the correction models')

In [ ]:
def train_run(tag, c, seed=0, use_image=True, cfg=TRAIN, prefix=None, DTR=None, w_tr=None, w_va=None):
    DTR = TR if DTR is None else DTR; w_tr = W_TR if w_tr is None else w_tr; w_va = W_VA if w_va is None else w_va
    ck = f'{OUT}/ckpt/{prefix or RUN}_{tag}.pt'
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = make_model(c, use_image)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    st = dict(epoch=0, best=float('inf'), bad=0, hist=[], best_state=None, done=False)
    if os.path.exists(ck):
        st = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
        if not st.get('done'): print(f'[{tag}] resuming from epoch {st["epoch"]}')
    loss_fn = nn.MSELoss()
    while not st.get('done'):
        t0 = time.time(); model.train(); tl = []
        for xv, img, cst, y in batches(DTR, w_tr, c, cfg['batch'], True, rng):
            loss = loss_fn(model(xv, img, cst), y); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval(); vs, vn = 0.0, 0
        with torch.no_grad():
            for xv, img, cst, y in batches(DTR, w_va, c, 2048, False):
                vs += loss_fn(model(xv, img, cst), y).item() * len(y); vn += len(y)
        v = vs / vn; st['hist'].append((float(np.mean(tl)), v)); st['epoch'] += 1
        if v < st['best']:
            st['best'], st['bad'] = v, 0
            st['best_state'] = {k: x.detach().clone() for k, x in model.state_dict().items()}
        else:
            st['bad'] += 1
        st['done'] = st['epoch'] >= cfg['max_epochs'] or st['bad'] >= cfg['patience']
        st['model'], st['opt'] = model.state_dict(), opt.state_dict()
        torch.save(st, ck)
        print(f'[{tag}] epoch {st["epoch"]:2d}  train {np.mean(tl):.4f}  val {v:.4f}  ({time.time()-t0:.0f}s)')
    model.load_state_dict(st['best_state']); model.eval()
    return model, st

@torch.no_grad()
def predict(model, D, w, c, mc_passes=0):
    if mc_passes: model.train()
    mean, std = [], []
    for xv, img, cst, _ in batches(D, w, c, 2048, False):
        if mc_passes:
            s = torch.stack([model(xv, img, cst) for _ in range(mc_passes)])
            mean.append(s.mean(0).cpu().numpy()); std.append(s.std(0).cpu().numpy())
        else:
            mean.append(model(xv, img, cst).cpu().numpy())
    model.eval()
    m = np.concatenate(mean) * PV_CAP
    return (m, np.concatenate(std) * PV_CAP) if mc_passes else m

In [ ]:
def cosine_betas(T, s=0.008):
    x = torch.linspace(0, T, T + 1)
    ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2; ac = ac / ac[0]
    return torch.clip(1 - ac[1:] / ac[:-1], 1e-4, 0.999)
BETAS = cosine_betas(DIFF['T']).to(device); ALPHAS = 1 - BETAS; ACP = torch.cumprod(ALPHAS, 0)

def t_embed(t, dim=16):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float()[:, None] * f[None]
    return torch.cat([torch.sin(a), torch.cos(a)], -1)

class DiffNet(nn.Module):
    def __init__(self, n_vec, img_ch=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(img_ch, 16, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4))
        self.vec = nn.Sequential(nn.Linear(n_vec, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.den = nn.Sequential(nn.Linear(1 + 16 + 32 * 16 + 32, 256), nn.SiLU(), nn.Linear(256, 128), nn.SiLU(), nn.Linear(128, 1))
    def context(self, xv, img):
        return torch.cat([self.conv(img).flatten(1), self.vec(xv)], 1)
    def denoise(self, z, t, ctx):
        return self.den(torch.cat([z[:, None], t_embed(t), ctx], 1)).squeeze(-1)

D_C = dict(SEL_C, resid=True)
y_tr_n = TR['pv'][W_TR + HORIZON] / PV_CAP
sp_tr_n = smart_base(TR, W_TR) / PV_CAP
NORM = {'resid': (0.0, float((y_tr_n - sp_tr_n).std())), 'direct': (float(y_tr_n.mean()), float(y_tr_n.std()))}
print('target scaling:', NORM)

def to_z(mode, y, xv):
    mu, sd = NORM[mode]
    return ((y - xv[:, -1]) if mode == 'resid' else (y - mu)) / sd

def from_z(mode, z, xv):
    mu, sd = NORM[mode]
    return (xv[:, -1:] + z * sd) if mode == 'resid' else (mu + z * sd)

def diff_loss(model, mode, xv, img, y, gen=None):
    z0 = to_z(mode, y, xv); B = len(z0)
    t = torch.randint(0, DIFF['T'], (B,), device=device, generator=gen)
    eps = torch.randn(B, device=device, generator=gen)
    zt = ACP[t].sqrt() * z0 + (1 - ACP[t]).sqrt() * eps
    return ((model.denoise(zt, t, model.context(xv, img)) - eps) ** 2).mean()

def train_diffusion(mode, seed=0, prefix=None):
    ck = f'{OUT}/ckpt/{prefix or RUN}_diffusion_{mode}' + ('' if seed == 0 else f'_seed{seed}') + '.pt'
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = DiffNet(n_vec(D_C), img_ch=6 if D_C.get('motion') else 3).to(device); opt = torch.optim.AdamW(model.parameters(), lr=DIFF['lr'], weight_decay=1e-4)
    st = dict(epoch=0, best=float('inf'), bad=0, hist=[], best_state=None, done=False)
    if os.path.exists(ck):
        st = torch.load(ck, map_location=device, weights_only=False); model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
    while not st['done']:
        t0 = time.time(); model.train(); tl = []
        for xv, img, cst, y in batches(TR, W_TR, D_C, DIFF['batch'], True, rng):
            loss = diff_loss(model, mode, xv, img, y); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval(); gen = torch.Generator(device=device).manual_seed(123); vs, vn = 0.0, 0
        with torch.no_grad():
            for xv, img, cst, y in batches(TR, W_VA, D_C, 4096, False):
                vs += diff_loss(model, mode, xv, img, y, gen).item() * len(y); vn += len(y)
        v = vs / vn; st['hist'].append((float(np.mean(tl)), v)); st['epoch'] += 1
        if v < st['best']:
            st['best'], st['bad'] = v, 0; st['best_state'] = {k: x.detach().clone() for k, x in model.state_dict().items()}
        else:
            st['bad'] += 1
        st['done'] = st['epoch'] >= DIFF['max_epochs'] or st['bad'] >= DIFF['patience']
        st['model'], st['opt'] = model.state_dict(), opt.state_dict(); torch.save(st, ck)
        print(f'[diffusion {mode}] epoch {st["epoch"]:2d}  train {np.mean(tl):.4f}  val {v:.4f}  ({time.time()-t0:.0f}s)')
    model.load_state_dict(st['best_state']); model.eval()
    return model, st

@torch.no_grad()
def diff_sample(model, mode, D, w, n=DIFF['n_samples'], bs=512, seed=0, clip=True):
    torch.manual_seed(seed); out = []
    for xv, img, cst, y in batches(D, w, D_C, bs, False):
        ctx = model.context(xv, img); B = len(ctx)
        ctx_r = ctx.repeat_interleave(n, 0); z = torch.randn(B * n, device=device)
        lo_z = to_z(mode, torch.zeros(B, device=device), xv).repeat_interleave(n, 0)
        hi_z = to_z(mode, torch.ones(B, device=device), xv).repeat_interleave(n, 0)
        for k in reversed(range(DIFF['T'])):
            tt = torch.full((B * n,), k, device=device, dtype=torch.long)
            eps = model.denoise(z, tt, ctx_r)
            ac, ac_prev = ACP[k], (ACP[k - 1] if k > 0 else torch.tensor(1.0, device=device))
            x0 = ((z - (1 - ac).sqrt() * eps) / ac.sqrt())
            if clip: x0 = torch.maximum(torch.minimum(x0, hi_z), lo_z)
            mean = (ac_prev.sqrt() * BETAS[k] / (1 - ac)) * x0 + (ALPHAS[k].sqrt() * (1 - ac_prev) / (1 - ac)) * z
            if k > 0:
                var = BETAS[k] * (1 - ac_prev) / (1 - ac)
                z = mean + var.sqrt() * torch.randn_like(z)
            else:
                z = x0
        yhat = from_z(mode, z.view(B, n), xv).clamp(0, 1) * PV_CAP
        out.append(yhat.cpu().numpy())
    return np.concatenate(out)

## 2. Predictions from notebooks 02 and 03

In [ ]:
VAL = W_VA
y_val = TR['pv'][VAL + HORIZON]
VPRED, VSIG = {}, {}
mem_t = {label: [] for label in TESTS}; mem_v = []
for s in ENSEMBLE_SEEDS:
    tag = 'A4' if s == 0 else f'A4_seed{s}'
    m, _ = train_run(tag, SEL_C, seed=s, prefix='feat')
    for label, w in TESTS.items(): mem_t[label].append(predict(m, TE, w, SEL_C))
    mem_v.append(predict(m, TR, VAL, SEL_C))
    if s == 0:
        PRED[('A4 single model', 'raw test')], PRED[('A4 single model', 'clean test')] = mem_t['raw test'][0], mem_t['clean test'][0]
        VPRED['A4 single model'] = mem_v[0]
        for label, w in TESTS.items():
            PRED[('MC dropout', label)], SIGMA[('MC dropout', label)] = predict(m, TE, w, SEL_C, mc_passes=MC_PASSES)
        VPRED['MC dropout'], VSIG['MC dropout'] = predict(m, TR, VAL, SEL_C, mc_passes=MC_PASSES)
    del m; torch.cuda.empty_cache()
for label in TESTS:
    st = np.stack(mem_t[label]); PRED[('deep ensemble', label)] = st.mean(0); SIGMA[('deep ensemble', label)] = st.std(0)
st = np.stack(mem_v); VPRED['deep ensemble'] = st.mean(0); VSIG['deep ensemble'] = st.std(0)
for n in ['A4 single model', 'deep ensemble', 'MC dropout']: score(n, notes='Day 2 checkpoints')

In [ ]:
R_C = dict(SEL_C, resid=True)
r_t = {label: [] for label in TESTS}; r_v = []
for s in RESID_SEEDS:
    m, _ = train_run(f'R4_seed{s}', R_C, seed=s, prefix='day3')
    for label, w in TESTS.items(): r_t[label].append(predict(m, TE, w, R_C))
    r_v.append(predict(m, TR, VAL, R_C)); del m; torch.cuda.empty_cache()
for label in TESTS:
    PRED[('correction CNN', label)] = r_t[label][0]
    st = np.stack(r_t[label]); PRED[('correction CNN, 3-seed ensemble', label)] = st.mean(0); SIGMA[('correction CNN, 3-seed ensemble', label)] = st.std(0)
VPRED['correction CNN'] = r_v[0]
st = np.stack(r_v); VPRED['correction CNN, 3-seed ensemble'] = st.mean(0); VSIG['correction CNN, 3-seed ensemble'] = st.std(0)

SAMPLES, VSAMPLES = {}, {}
for mode, name in [('resid', 'diffusion (correction)'), ('direct', 'diffusion (direct)')]:
    p = f'{OUT}/day3_samples_{mode}.npz'
    if not os.path.exists(p): print('missing', p); continue
    z = np.load(p); SAMPLES[(name, 'raw test')], SAMPLES[(name, 'clean test')], VSAMPLES[name] = z['raw'], z['clean'], z['val']
    for label in TESTS: PRED[(name, label)] = SAMPLES[(name, label)].mean(1); SIGMA[(name, label)] = SAMPLES[(name, label)].std(1)
    VPRED[name] = VSAMPLES[name].mean(1); VSIG[name] = VSAMPLES[name].std(1)
print('loaded:', sorted({n for n, _ in PRED}))

In [ ]:
from scipy.stats import norm
VPRED['smart persistence (capped)'] = smart_base(TR, VAL)
GAUSS = ['deep ensemble', 'MC dropout', 'correction CNN, 3-seed ensemble']
SAMP = [n for n in ['diffusion (correction)', 'diffusion (direct)'] if n in VPRED]
POINT = ['smart persistence (capped)', 'correction CNN']

def sig_eff(name, sig):
    '''Spread plus a floor (median validation spread), so near-zero spreads cannot blow up the conformal factor.'''
    return np.asarray(sig) + float(np.median(VSIG[name]))

def conf_q(scores, level):
    n = len(scores); return float(np.quantile(scores, min(1.0, math.ceil((n + 1) * level) / n)))

def interval(name, label, level=1 - ALPHA, conformal=True):
    '''Return (lo, hi) on the test set for a two-sided interval at this level.'''
    a = (1 - level) / 2
    if name in GAUSS:
        mu = PRED[(name, label)]
        if not conformal:
            sg = np.maximum(SIGMA[(name, label)], 1e-6); k = norm.ppf(1 - a); return mu - k * sg, mu + k * sg
        sg = sig_eff(name, SIGMA[(name, label)])
        k = conf_q(np.abs(y_val - VPRED[name]) / sig_eff(name, VSIG[name]), level)
        return mu - k * sg, mu + k * sg
    if name in SAMP:
        s = SAMPLES[(name, label)]; lo, hi = np.quantile(s, a, 1), np.quantile(s, 1 - a, 1)
        if conformal:
            vl, vh = np.quantile(VSAMPLES[name], a, 1), np.quantile(VSAMPLES[name], 1 - a, 1)
            q = conf_q(np.maximum(vl - y_val, y_val - vh), level); lo, hi = lo - q, hi + q
        return lo, hi
    q = conf_q(np.abs(y_val - VPRED[name]), level)
    return PRED[(name, label)] - q, PRED[(name, label)] + q

def crps_samples(s, y, chunk=2000):
    tot = 0.0
    for i in range(0, len(y), chunk):
        a, b = s[i:i + chunk], y[i:i + chunk]
        tot += (np.abs(a - b[:, None]).mean(1) - 0.5 * np.abs(a[:, :, None] - a[:, None, :]).mean((1, 2))).sum()
    return tot / len(y)

def crps_gauss(mu, sg, y):
    sg = np.maximum(sg, 1e-6); z = (y - mu) / sg
    return float(np.mean(sg * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))))

In [ ]:
def day_classes(D):
    day = D['zen'] < 80
    df = pd.DataFrame({'d': D['ld'][day], 'pv': D['pv'][day], 'cs': D['cs'][day], 'k': D['k'][day]})
    g = df.groupby('d')
    mean_k = g.pv.sum() / g.cs.sum().clip(lower=1e-6)
    var_k = g.k.apply(lambda s: np.abs(np.diff(s.values)).mean() if len(s) > 1 else np.nan)
    cls = pd.Series(np.where((var_k < 0.01) & (mean_k > 0.7), 'clear', np.where(mean_k < 0.45, 'overcast', 'partly cloudy')), index=mean_k.index)
    return cls, var_k
KINDS = ['clear', 'partly cloudy', 'overcast']
CLS, VARK = day_classes(TE); CLS_TR, _ = day_classes(TR)
label = 'clean test'; w = TESTS[label]; y = TE['pv'][w + HORIZON]
wcls = pd.Series(TE['ld'][w]).map(CLS).values; vcls = pd.Series(TR['ld'][VAL]).map(CLS_TR).values

def recent_var(D, ww):
    '''Std of the clear-sky index over the last 15 minutes: known at forecast time.'''
    return D['k'][ww[:, None] + OFF].std(1)

def boot_ci(name, ref='smart persistence', label='clean test', n=2000, seed=0, preds=None, ref_preds=None, ww=None, yy=None):
    ww = TESTS[label] if ww is None else ww; yy = TE['pv'][ww + HORIZON] if yy is None else yy; days = TE['ld'][ww]
    pm = PRED[(name, label)] if preds is None else preds; pr = PRED[(ref, label)] if ref_preds is None else ref_preds
    em, er = pm - yy, pr - yy
    g = pd.DataFrame({'d': days, 'sm': em ** 2, 'sr': er ** 2, 'am': np.abs(em), 'ar': np.abs(er), 'n': 1}).groupby('d').sum()
    idx = np.random.default_rng(seed).integers(0, len(g), (n, len(g)))
    t = lambda c: g[c].values[idx].sum(1)
    sk_r = 1 - np.sqrt(t('sm') / t('n')) / np.sqrt(t('sr') / t('n')); sk_m = 1 - t('am') / t('ar')
    return dict(model=name, vs=ref, rmse_skill=1 - np.sqrt((em ** 2).mean()) / np.sqrt((er ** 2).mean()),
                rmse_skill_lo=np.percentile(sk_r, 2.5), rmse_skill_hi=np.percentile(sk_r, 97.5),
                mae_skill=1 - np.abs(em).mean() / np.abs(er).mean(), mae_skill_lo=np.percentile(sk_m, 2.5), mae_skill_hi=np.percentile(sk_m, 97.5))

## 3. Weather mix of validation and test days

In [ ]:
mix = pd.DataFrame({'validation days': pd.Series(vcls).value_counts(normalize=True),
                    'test days (clean)': pd.Series(wcls).value_counts(normalize=True)}).reindex(KINDS)
rv_val, rv_te = recent_var(TR, VAL), recent_var(TE, w)
print('Share of forecast windows by weather type:'); print((100 * mix).round(1).to_string())
print(f'\nRecent variability (std of clear-sky index, last 15 min): validation median {np.median(rv_val):.3f}, 90th pct {np.percentile(rv_val, 90):.3f}'
      f' | test median {np.median(rv_te):.3f}, 90th pct {np.percentile(rv_te, 90):.3f}')
s_val, s_te = metrics(VPRED['smart persistence (capped)'], y_val), metrics(PRED[('smart persistence (capped)', label)], y)
print(f'Smart persistence RMSE: validation {s_val[0]:.3f} vs test {s_te[0]:.3f}  (higher on test = test days are harder)')
mix.to_csv(f'{OUT}/day4_weather_mix.csv')

## 4. Adaptive calibration
Split conformal calibration done separately within groups of recent variability (standard deviation of the clear-sky index over the last 15 minutes). Group boundaries are set on validation days.

In [ ]:
EDGES = np.quantile(rv_val, VAR_BIN_QUANTILES)
BV, BT = np.digitize(rv_val, EDGES), np.digitize(rv_te, EDGES)
print('validation windows per group:', np.bincount(BV), '| test windows per group:', np.bincount(BT, minlength=len(EDGES) + 1))

def interval_ad(name, level=1 - ALPHA, adaptive=True):
    a = (1 - level) / 2; bv = BV if adaptive else np.zeros_like(BV); bt = BT if adaptive else np.zeros_like(BT)
    lo, hi = np.empty(len(y)), np.empty(len(y))
    if name in GAUSS:
        mu, sg = PRED[(name, label)], sig_eff(name, SIGMA[(name, label)]); sv = np.abs(y_val - VPRED[name]) / sig_eff(name, VSIG[name])
        for b in np.unique(bt):
            k = conf_q(sv[bv == b], level); m = bt == b; lo[m], hi[m] = mu[m] - k * sg[m], mu[m] + k * sg[m]
    elif name in SAMP:
        s = SAMPLES[(name, label)]; l0, h0 = np.quantile(s, a, 1), np.quantile(s, 1 - a, 1)
        vl, vh = np.quantile(VSAMPLES[name], a, 1), np.quantile(VSAMPLES[name], 1 - a, 1); sv = np.maximum(vl - y_val, y_val - vh)
        for b in np.unique(bt):
            q = conf_q(sv[bv == b], level); m = bt == b; lo[m], hi[m] = l0[m] - q, h0[m] + q
    else:
        p = PRED[(name, label)]; sv = np.abs(y_val - VPRED[name])
        for b in np.unique(bt):
            q = conf_q(sv[bv == b], level); m = bt == b; lo[m], hi[m] = p[m] - q, p[m] + q
    return lo, hi

def lower_ad(name, level=RESERVE_LEVEL, adaptive=True):
    bv = BV if adaptive else np.zeros_like(BV); bt = BT if adaptive else np.zeros_like(BT); lb = np.empty(len(y))
    if name in GAUSS:
        sv = (VPRED[name] - y_val) / sig_eff(name, VSIG[name]); mu, sg = PRED[(name, label)], sig_eff(name, SIGMA[(name, label)])
        for b in np.unique(bt): m = bt == b; lb[m] = mu[m] - conf_q(sv[bv == b], level) * sg[m]
    elif name in SAMP:
        sv = np.quantile(VSAMPLES[name], 1 - level, 1) - y_val; l0 = np.quantile(SAMPLES[(name, label)], 1 - level, 1)
        for b in np.unique(bt): m = bt == b; lb[m] = l0[m] - conf_q(sv[bv == b], level)
    else:
        sv = VPRED[name] - y_val; p = PRED[(name, label)]
        for b in np.unique(bt): m = bt == b; lb[m] = p[m] - conf_q(sv[bv == b], level)
    return np.maximum(lb, 0)

METHODS = POINT + GAUSS + SAMP
rows, rrows = [], []
for name in METHODS:
    for ver, ad in [('Day 3 (one group)', False), ('adaptive (by recent variability)', True)]:
        lo, hi = interval_ad(name, adaptive=ad); cov = (y >= lo) & (y <= hi)
        r = dict(model=name, version=ver, coverage=cov.mean(), width=(hi - lo).mean())
        for k in KINDS:
            m = wcls == k; r[f'cov_{k}'] = cov[m].mean() if m.any() else np.nan; r[f'width_{k}'] = (hi - lo)[m].mean() if m.any() else np.nan
        rows.append(r)
        lb = lower_ad(name, adaptive=ad); res = np.maximum(PRED[(name, label)] - lb, 0)
        rr = dict(model=name, version=ver, shortfall_rate=(y < lb).mean(), mean_reserve_kw=res.mean())
        for k in KINDS:
            m = wcls == k; rr[f'shortfall_{k}'] = (y[m] < lb[m]).mean() if m.any() else np.nan; rr[f'reserve_{k}'] = res[m].mean() if m.any() else np.nan
        rrows.append(rr)
CAL4 = pd.DataFrame(rows); RES4 = pd.DataFrame(rrows)
CAL4.to_csv(f'{OUT}/day4_adaptive_calibration.csv', index=False); RES4.to_csv(f'{OUT}/day4_adaptive_reserve.csv', index=False)
pd.set_option('display.width', 220)
print('90% intervals (target coverage 0.90):'); print(CAL4.round(3).to_string(index=False))
print(f'\nReserve for {RESERVE_LEVEL:.0%} reliability (target shortfall {1-RESERVE_LEVEL:.0%}):'); print(RES4.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ad = CAL4[CAL4.version.str.startswith('adaptive')].set_index('model').loc[METHODS]
g3 = CAL4[CAL4.version.str.startswith('Day 3')].set_index('model').loc[METHODS]
bw = 0.8 / len(METHODS)
for i, name in enumerate(METHODS):
    axes[0].bar(np.arange(3) + i * bw - 0.4 + bw / 2, [ad.loc[name, f'cov_{k}'] for k in KINDS], bw, label=name)
    axes[0].scatter(np.arange(3) + i * bw - 0.4 + bw / 2, [g3.loc[name, f'cov_{k}'] for k in KINDS], color='k', s=10, zorder=3)
axes[0].axhline(0.9, color='0.3', ls='--', lw=1); axes[0].set_ylim(0.4, 1.02)
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(KINDS); axes[0].set_ylabel('coverage of 90% interval')
axes[0].set_title('Adaptive calibration (bars) vs Day 3 (dots)'); axes[0].legend(fontsize=6)
ra = RES4[RES4.version.str.startswith('adaptive')].set_index('model').loc[METHODS]
for i, name in enumerate(METHODS):
    axes[1].bar(np.arange(3) + i * bw - 0.4 + bw / 2, [ra.loc[name, f'reserve_{k}'] for k in KINDS], bw, label=name)
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(KINDS); axes[1].set_ylabel('mean reserve (kW)')
axes[1].set_title(f'Reserve for {RESERVE_LEVEL:.0%} reliability, adaptive calibration')
fig.tight_layout(); fig.savefig(f'{OUT}/figs/day4_adaptive_calibration.png', dpi=150); plt.show()

## 5. A5 against A4 on validation loss

In [ ]:
def ck_best(p): return float(torch.load(p, map_location='cpu', weights_only=False)['best'])
a4 = [ck_best(f'{OUT}/ckpt/feat_A4.pt' if s == 0 else f'{OUT}/ckpt/feat_A4_seed{s}.pt') for s in ENSEMBLE_SEEDS]
a5 = [ck_best(f'{OUT}/ckpt/feat_A5.pt' if s == 0 else f'{OUT}/ckpt/day3_A5_seed{s}.pt') for s in A5_SEEDS]
gap = np.mean(a4) - np.mean(a5); se = np.sqrt(np.var(a4, ddof=1) / len(a4) + np.var(a5, ddof=1) / len(a5))
A5_WINS = bool(gap > 2 * se) if FORCE_A5 is None else FORCE_A5
v = pd.DataFrame({'config': ['A4 + PV variability', 'A5 + cloud motion'], 'seeds': [len(a4), len(a5)],
                  'val_mse_mean': [np.mean(a4), np.mean(a5)], 'val_mse_std': [np.std(a4, ddof=1), np.std(a5, ddof=1)]})
print(v.to_string(index=False)); v.to_csv(f'{OUT}/day4_a5_validation.csv', index=False)
print(f'\nGap (A4 - A5) = {gap:.6f}, about {gap / se:.1f} standard errors. A5 wins on validation: {A5_WINS}')

## 6. Inference time

In [ ]:
NT = TESTS['clean test'][:2048]
def sync():
    if device == 'cuda': torch.cuda.synchronize()
def timed(fn, reps=3):
    fn(); sync(); t0 = time.time()
    for _ in range(reps): fn()
    sync(); return (time.time() - t0) / reps / len(NT) * 1000
T = {}
T['smart persistence'] = timed(lambda: smart_base(TE, NT))
ens = [train_run('A4' if s == 0 else f'A4_seed{s}', SEL_C, seed=s, prefix='feat')[0] for s in ENSEMBLE_SEEDS]
T['single CNN'] = timed(lambda: predict(ens[0], TE, NT, SEL_C))
T[f'deep ensemble ({len(ens)} CNNs)'] = timed(lambda: [predict(m, TE, NT, SEL_C) for m in ens])
T[f'MC dropout ({MC_PASSES} passes)'] = timed(lambda: predict(ens[0], TE, NT, SEL_C, mc_passes=MC_PASSES))
del ens; torch.cuda.empty_cache()
DMODEL, _ = train_diffusion('resid', prefix='day3')
T[f'diffusion ({DIFF["n_samples"]} samples x {DIFF["T"]} steps)'] = timed(lambda: diff_sample(DMODEL, 'resid', TE, NT), reps=1)
tt = pd.DataFrame({'method': list(T), 'ms_per_forecast': list(T.values())})
tt['relative_to_single_CNN'] = tt.ms_per_forecast / T['single CNN']
print(tt.round(3).to_string(index=False)); tt.to_csv(f'{OUT}/day4_timing.csv', index=False)
print('Timed on the same 2048 test windows on this GPU, including data loading for every method.')

## 7. Diffusion with and without clipping during sampling

In [ ]:
p = f'{OUT}/day4_samples_noclip_clean.npz'
if os.path.exists(p): S_NC = np.load(p)['s']
else:
    S_NC = diff_sample(DMODEL, 'resid', TE, w, clip=False); np.savez_compressed(p, s=S_NC)
def prob_scores(S):
    lo, hi = np.quantile(S, 0.05, 1), np.quantile(S, 0.95, 1); r, m = metrics(S.mean(1), y)
    return dict(rmse=r, mae=m, crps=crps_samples(S, y), coverage_90_raw=float(np.mean((y >= lo) & (y <= hi))), width_90_raw=float(np.mean(hi - lo)))
CL = pd.DataFrame([dict(version='with physical limits (Day 3)', **prob_scores(SAMPLES[('diffusion (correction)', label)])),
                   dict(version='without limits during sampling', **prob_scores(S_NC))])
print(CL.round(3).to_string(index=False)); CL.to_csv(f'{OUT}/day4_constraint_ablation.csv', index=False)
print('Both versions clip the final output to 0..cap; only the per-step limit differs.')

## 8. Effect of the uncorrected clock on a trained model

In [ ]:
def buggy_copy(D):
    B = dict(D); cs, zen, azi = solar_terms(D['ts'])
    B['cs_raw'] = np.nan_to_num(cs); B['zen'] = np.nan_to_num(zen, nan=180.0); B['azi'] = np.nan_to_num(azi)
    B['cs'] = np.minimum(B['cs_raw'], PV_CAP).astype(np.float32)
    B['k'] = np.clip(B['pv'] / np.maximum(B['cs'], 1.0), 0, 1.5).astype(np.float32)
    return B
TRb, TEb = buggy_copy(TR), buggy_copy(TE)
mb, _ = train_run('R4_buggyclock', R_C, seed=0, DTR=TRb)
rows = []
for lab, ww in TESTS.items():
    yy = TE['pv'][ww + HORIZON]; ref = metrics(PRED[('smart persistence', lab)], yy)
    for name, pr in [('smart persistence, buggy clock', smart_base(TEb, ww)), ('smart persistence, fixed clock', smart_base(TE, ww)),
                     ('correction CNN, buggy clock', predict(mb, TEb, ww, R_C)), ('correction CNN, fixed clock (Day 3)', PRED[('correction CNN', lab)])]:
        r, m = metrics(pr, yy)
        rows.append(dict(test_set=lab, model=name, rmse=r, mae=m, skill_rmse_vs_fixed_smart=1 - r / ref[0], skill_mae_vs_fixed_smart=1 - m / ref[1]))
del mb; torch.cuda.empty_cache()
BUG = pd.DataFrame(rows); print(BUG.round(3).to_string(index=False)); BUG.to_csv(f'{OUT}/day4_bug_ablation.csv', index=False)

## 9. Forecast horizon

In [ ]:
TR_GOOD, TE_GOOD = P['tr_good'], P['te_good']
def windows_h(D, good, h):
    ts_min = D['ts'].values.astype('datetime64[s]').astype(np.int64) / 60.0
    d = np.diff(ts_min); cbad = np.r_[0, np.cumsum((d > 1 + 1e-6) | (d <= 0))]
    t = np.arange(INPUT_LEN - 1, len(ts_min) - h); start, end = t - (INPUT_LEN - 1), t + h
    ok = (cbad[end] - cbad[start]) == 0
    cg = np.r_[0, np.cumsum(~good)]; ok &= (cg[end + 1] - cg[start]) == 0
    return t[ok]
assert len(windows_h(TR, TR_GOOD, 15)) == len(TR['w_clean']), 'window rebuild does not match Day 1'

HROWS = []
hp = f'{OUT}/day4_horizons.csv'
for h in sorted(set(HORIZONS + [15])):
    HORIZON = h
    wtr_all, wte = windows_h(TR, TR_GOOD, h), windows_h(TE, TE_GOOD, h)
    wtr, wva = split_by_days(TR, wtr_all, TRAIN['val_frac_days'])
    yy = TE['pv'][wte + h]
    naive = TE['pv'][wte]; smart = np.clip(TE['pv'][wte] / np.maximum(TE['cs_raw'][wte], 1e-3) * TE['cs_raw'][wte + h], 0, None)
    ref = metrics(smart, yy)
    mi, _ = (train_run('R4_seed0', R_C, seed=0, prefix='day3') if h == 15 else train_run(f'R4_h{h}', R_C, seed=0, w_tr=wtr, w_va=wva))
    mp, _ = train_run(f'R4pv_h{h}', R_C, seed=0, use_image=False, w_tr=wtr, w_va=wva)
    for name, pr in [('naive persistence', naive), ('smart persistence', smart), ('correction CNN, image + PV', predict(mi, TE, wte, R_C)),
                     ('correction CNN, PV only', predict(mp, TE, wte, R_C))]:
        r, m = metrics(pr, yy)
        HROWS.append(dict(horizon_min=h, model=name, n=len(yy), rmse=r, mae=m, skill_rmse=1 - r / ref[0], skill_mae=1 - m / ref[1]))
    del mi, mp; torch.cuda.empty_cache()
    pd.DataFrame(HROWS).to_csv(hp, index=False)
HORIZON = 15
HZ = pd.DataFrame(HROWS); print(HZ.round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, lab in [(axes[0], 'skill_rmse', 'RMSE skill vs smart persistence (%)'), (axes[1], 'skill_mae', 'MAE skill vs smart persistence (%)')]:
    for name in ['correction CNN, image + PV', 'correction CNN, PV only']:
        q = HZ[HZ.model == name]; ax.plot(q.horizon_min, 100 * q[col], marker='o', label=name)
    ax.axhline(0, color='0.5', lw=.8); ax.set_xlabel('forecast horizon (min)'); ax.set_ylabel(lab); ax.legend(fontsize=8)
fig.suptitle('Value of sky images by forecast horizon (clean test)'); fig.tight_layout()
fig.savefig(f'{OUT}/figs/day4_horizons.png', dpi=150); plt.show()

## 10. Diffusion trained for longer, three seeds

In [ ]:
DIFF_SAVE = dict(DIFF); DIFF.update(DIFF_LONG)
DS, POOL, VPOOL = [], {}, {}
for mode in DIFF_MODES:
    nm = 'correction' if mode == 'resid' else 'direct'
    for s in DIFF_SEEDS:
        p = f'{OUT}/day4_samples_long_{mode}_seed{s}.npz'
        if os.path.exists(p):
            z = np.load(p); S, VS, ep = z['clean'], z['val'], int(z['epochs'])
        else:
            m, st = train_diffusion(mode, seed=s, prefix='day4long'); ep = st['epoch']
            S = diff_sample(m, mode, TE, w, seed=s); VS = diff_sample(m, mode, TR, VAL, seed=s)
            np.savez_compressed(p, clean=S, val=VS, epochs=ep); del m; torch.cuda.empty_cache()
        DS.append(dict(version=f'diffusion ({nm}), longer training', seed=s, epochs=ep, **prob_scores(S)))
        POOL.setdefault(nm, []).append(S); VPOOL.setdefault(nm, []).append(VS)
        pd.DataFrame(DS).to_csv(f'{OUT}/day4_diffusion_seeds.csv', index=False)
DIFF.clear(); DIFF.update(DIFF_SAVE)
for mode, nm in [('resid', 'correction'), ('direct', 'direct')]:
    DS.append(dict(version=f'diffusion ({nm}), Day 3 (15 epochs)', seed=0, epochs=15, **prob_scores(SAMPLES[(f'diffusion ({nm})', label)])))
for nm in POOL:
    DS.append(dict(version=f'diffusion ({nm}), 3 seeds combined', seed=-1, epochs=np.nan, **prob_scores(np.concatenate(POOL[nm], 1))))
DSD = pd.DataFrame(DS); DSD.to_csv(f'{OUT}/day4_diffusion_seeds.csv', index=False)
print(DSD.round(3).to_string(index=False))
print('\nmean ± std across seeds (longer training):')
print(DSD[(DSD.seed >= 0) & DSD.version.str.contains('longer')].groupby('version')[['rmse', 'mae', 'crps', 'coverage_90_raw']].agg(['mean', 'std']).round(3).to_string())
print(f"Deep ensemble CRPS for reference: {crps_gauss(PRED[('deep ensemble', label)], SIGMA[('deep ensemble', label)], y):.3f}")

for nm in POOL:
    name = f'diffusion ({nm}), longer, 3 seeds'
    SAMPLES[(name, label)] = np.concatenate(POOL[nm], 1); VSAMPLES[name] = np.concatenate(VPOOL[nm], 1)
    PRED[(name, label)] = SAMPLES[(name, label)].mean(1); VPRED[name] = VSAMPLES[name].mean(1); VSIG[name] = VSAMPLES[name].std(1)
    SAMP.append(name)
rows, rrows = [], []
for name in [n for n in SAMP if 'longer' in n]:
    for ver, ad in [('Day 3 (one group)', False), ('adaptive (by recent variability)', True)]:
        lo, hi = interval_ad(name, adaptive=ad); cov = (y >= lo) & (y <= hi)
        r = dict(model=name, version=ver, coverage=cov.mean(), width=(hi - lo).mean())
        for k in KINDS:
            m = wcls == k; r[f'cov_{k}'] = cov[m].mean() if m.any() else np.nan; r[f'width_{k}'] = (hi - lo)[m].mean() if m.any() else np.nan
        rows.append(r)
        lb = lower_ad(name, adaptive=ad); res = np.maximum(PRED[(name, label)] - lb, 0)
        rr = dict(model=name, version=ver, shortfall_rate=(y < lb).mean(), mean_reserve_kw=res.mean())
        for k in KINDS:
            m = wcls == k; rr[f'shortfall_{k}'] = (y[m] < lb[m]).mean() if m.any() else np.nan; rr[f'reserve_{k}'] = res[m].mean() if m.any() else np.nan
        rrows.append(rr)
pd.DataFrame(rows).to_csv(f'{OUT}/day4_long_diffusion_calibration.csv', index=False)
pd.DataFrame(rrows).to_csv(f'{OUT}/day4_long_diffusion_reserve.csv', index=False)
print(pd.DataFrame(rows).round(3).to_string(index=False)); print(pd.DataFrame(rrows).round(3).to_string(index=False))

## 11. Models with cloud motion

In [ ]:
if not (RUN_A5_MODELS and A5_WINS):
    print('Skipped: A5 did not beat A4 on validation loss, so its test-set gain stays an exploratory finding.')
else:
    R5_C = dict(A5_C, resid=True); r5 = []
    for s in RESID_SEEDS:
        m, _ = train_run(f'R5_seed{s}', R5_C, seed=s); r5.append(predict(m, TE, w, R5_C)); del m; torch.cuda.empty_cache()
    p5 = np.mean(r5, 0); rows = []
    for name, pr in [('correction CNN + cloud motion, 3 seeds', p5), ('correction CNN, 3 seeds (Day 3)', PRED[('correction CNN, 3-seed ensemble', label)])]:
        rows.append(boot_ci(name, preds=pr, ref_preds=PRED[('smart persistence', label)]))
    rows.append(boot_ci('correction CNN + cloud motion, 3 seeds', ref='correction CNN, 3 seeds (Day 3)', preds=p5, ref_preds=PRED[('correction CNN, 3-seed ensemble', label)]))
    A5T = pd.DataFrame(rows); print(A5T.round(3).to_string(index=False)); A5T.to_csv(f'{OUT}/day4_a5_models.csv', index=False)
    p = f'{OUT}/day4_samples_diffA5_clean.npz'
    D_C_SAVE = D_C; D_C = dict(A5_C, resid=True); DIFF_SAVE = dict(DIFF); DIFF.update(DIFF_LONG)
    if os.path.exists(p): S5 = np.load(p)['s']
    else:
        m, _ = train_diffusion('resid', prefix='day4A5'); S5 = diff_sample(m, 'resid', TE, w); np.savez_compressed(p, s=S5); del m
    D_C = D_C_SAVE; DIFF.clear(); DIFF.update(DIFF_SAVE)
    d5 = pd.DataFrame([dict(model='diffusion (correction), A4 features, longer training, seed 0', **prob_scores(POOL['correction'][0])),
                       dict(model='diffusion (correction), + cloud motion, longer training, seed 0', **prob_scores(S5))])
    print(d5.round(3).to_string(index=False)); d5.to_csv(f'{OUT}/day4_a5_diffusion.csv', index=False)

In [ ]:
outs = sorted(f for f in os.listdir(OUT) if f.startswith('day4_') and f.endswith('.csv'))
figs = sorted(f for f in os.listdir(f'{OUT}/figs') if f.startswith('day4_'))
print('Done. Saved CSVs:', outs); print('and figures:', figs)